# 04. Equilíbrio e rerandomização

A randomização equilibra as covariáveis **em média**, mas um sorteio específico pode sair desequilibrado. Este notebook mostra como **medir** o equilíbrio (SMD) e como a rerandomização melhora um sorteio ruim antes de rodar o experimento.

## Medir o equilíbrio com SMD

A diferença de médias padronizada (SMD) compara tratados e controles em cada covariável. Regra prática: `|SMD| > 0.1` indica desequilíbrio relevante.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.design.balance import check_balance
from skxperiments.design.crd import CRD

rng = np.random.default_rng(3)
n = 100
df = pd.DataFrame({"x1": rng.normal(size=n), "x2": rng.normal(size=n)})

crd = CRD(p=0.5, seed=3).randomize(df)
check_balance(crd)[["covariate", "smd"]]

## Rerandomização

`ReRandomizedCRD` sorteia de novo até a distância de Mahalanobis entre os grupos ficar abaixo de um limiar. O resultado é um sorteio com SMDs menores.

In [ ]:
from scipy.stats import chi2

from skxperiments.design.rerandomized_crd import ReRandomizedCRD

threshold = float(chi2.ppf(0.05, df=2))   # aceita ~5% dos sorteios mais equilibrados
rerand = ReRandomizedCRD(
    covariates=["x1", "x2"], threshold=threshold, p=0.5, seed=3, max_attempts=10000
).randomize(df)
check_balance(rerand)[["covariate", "smd"]]

## Relatório e gráfico

`BalanceReport` resume os SMDs e sinaliza desequilíbrios; `plot_balance` desenha o Love plot (precisa de matplotlib, no extra `viz`).

In [ ]:
from skxperiments.diagnostics import BalanceReport
from skxperiments.reporting import plot_balance

report = BalanceReport().run(rerand)
print("Desequilibradas:", report.imbalanced)
ax = plot_balance(report)
ax.figure

## O que aprendemos

- SMD mede o equilíbrio de cada covariável; `|SMD| > 0.1` é um sinal de alerta.
- A rerandomização rejeita sorteios desequilibrados **antes** de coletar dados, melhorando o balanço sem violar a randomização.
- `BalanceReport` e `plot_balance` deixam isso visível num relatório.

**Próximo:** `05. Blocagem` garante equilíbrio em variáveis categóricas conhecidas.